# Week 6: Channel-Based Sampling — TED and TEDx

This notebook initiates a pivot to channel-based sampling by collecting all videos
from the official TED and TEDx YouTube channels. The goal is to construct a
systematic dataset of expert discourse on YouTube while reusing the sentiment
analysis pipeline developed in earlier weeks.


In [1]:
!git clone https://github.com/mustafayubk/SOSC314_Project.git
%cd SOSC314_Project
!ls


fatal: destination path 'SOSC314_Project' already exists and is not an empty directory.
/content/SOSC314_Project
 config   docs	      README.md  'Week 2_Figure.png'
 data	  notebooks   scripts	  Week_3_Figure.png


In [2]:
import os

os.makedirs("notebooks/week6", exist_ok=True)


## Step 1: Retrieve all videos from TED and TEDx channels

This step uses the YouTube Data API to collect metadata for all videos
uploaded by the official TED and TEDx channels. This provides a systematic,
channel-based sampling frame for expert discourse on YouTube.


In [3]:
!pip -q install google-api-python-client pandas tqdm


In [4]:
import os
import pandas as pd
from tqdm import tqdm
from googleapiclient.discovery import build


In [6]:
from google.colab import userdata


In [7]:
API_KEY = userdata.get("YOUTUBE_API_KEY") or userdata.get("YOUTUBE")

print("Has key?", API_KEY is not None)

if not API_KEY:
    raise ValueError("YouTube API key not found. In Colab Secrets, create YOUTUBE_API_KEY (or YOUTUBE) and enable Notebook access.")

youtube = build("youtube", "v3", developerKey=API_KEY)
print("YouTube client built ✅")


Has key? True
YouTube client built ✅


In [8]:
CHANNELS = {
    "TED": "UCAuUUnT6oDeKwE6v1NGQxug",
    "TEDx": "UCsT0YIqwnpJCM-mx7-gSA4Q"
}


In [9]:
def get_all_videos_from_channel(channel_id, channel_name):
    videos = []
    next_page = None

    while True:
        request = youtube.search().list(
            part="snippet",
            channelId=channel_id,
            maxResults=50,
            pageToken=next_page,
            type="video",
            order="date"
        )
        response = request.execute()

        for item in response["items"]:
            videos.append({
                "video_id": item["id"]["videoId"],
                "title": item["snippet"]["title"],
                "channel": channel_name,
                "published_at": item["snippet"]["publishedAt"]
            })

        next_page = response.get("nextPageToken")
        if not next_page:
            break

    return videos


In [10]:
all_videos = []

for name, cid in CHANNELS.items():
    print(f"Collecting videos from {name}...")
    vids = get_all_videos_from_channel(cid, name)
    print(f"  → {len(vids)} videos found")
    all_videos.extend(vids)


  → 10 videos found
  → 100 videos found


In [11]:
def get_uploads_playlist_id(channel_id: str) -> str:
    resp = youtube.channels().list(
        part="contentDetails",
        id=channel_id
    ).execute()
    items = resp.get("items", [])
    if not items:
        raise ValueError(f"No channel found for channel_id={channel_id}")
    return items[0]["contentDetails"]["relatedPlaylists"]["uploads"]

for name, cid in CHANNELS.items():
    upl = get_uploads_playlist_id(cid)
    print(name, "uploads playlist:", upl)


TED uploads playlist: UUAuUUnT6oDeKwE6v1NGQxug
TEDx uploads playlist: UUsT0YIqwnpJCM-mx7-gSA4Q


In [12]:
def get_all_videos_from_uploads_playlist(uploads_playlist_id: str, channel_name: str):
    videos = []
    page_token = None

    while True:
        resp = youtube.playlistItems().list(
            part="snippet,contentDetails",
            playlistId=uploads_playlist_id,
            maxResults=50,
            pageToken=page_token
        ).execute()

        for item in resp.get("items", []):
            # playlistItems gives videoId + publishedAt inside contentDetails
            vid = item["contentDetails"]["videoId"]
            published_at = item["contentDetails"].get("videoPublishedAt") or item["snippet"].get("publishedAt")
            title = item["snippet"]["title"]

            videos.append({
                "video_id": vid,
                "title": title,
                "channel": channel_name,
                "published_at": published_at
            })

        page_token = resp.get("nextPageToken")
        if not page_token:
            break

    return videos

all_videos = []
for name, cid in CHANNELS.items():
    uploads = get_uploads_playlist_id(cid)
    print(f"Collecting ALL uploads from {name}…")
    vids = get_all_videos_from_uploads_playlist(uploads, name)
    print(f"  → {len(vids)} videos found")
    all_videos.extend(vids)

print("TOTAL videos:", len(all_videos))


  → 5491 videos found
  → 20000 videos found
TOTAL videos: 25491


In [13]:
import pandas as pd

videos_df = pd.DataFrame(all_videos)
videos_df["published_at"] = pd.to_datetime(videos_df["published_at"], errors="coerce")

print(videos_df.groupby("channel")["video_id"].nunique())
print("Earliest by channel:")
print(videos_df.sort_values("published_at").groupby("channel").head(1)[["channel","published_at","title"]])
print("Latest by channel:")
print(videos_df.sort_values("published_at").groupby("channel").tail(1)[["channel","published_at","title"]])


channel
TED      5491
TEDx    20000
Name: video_id, dtype: int64
Earliest by channel:
      channel              published_at  \
5490      TED 2006-12-25 17:58:08+00:00   
25490    TEDx 2025-05-01 15:30:39+00:00   

                                                   title  
5490                If I controlled the Internet | Rives  
25490  Why you should make a ‘done’ list | Francis Ru...  
Latest by channel:
     channel              published_at  \
5491    TEDx 2026-02-08 14:30:11+00:00   
0        TED 2026-02-08 19:00:41+00:00   

                                                  title  
5491  How do non-living things ‘evolve’? | Michael W...  
0        What if plastic didn’t last forever? #TEDTalks  


In [14]:
import os
import pandas as pd

# Create folder if it doesn't exist
os.makedirs("data/raw/week6", exist_ok=True)

videos_df = pd.DataFrame(all_videos)

# (Optional but recommended) remove any duplicates just in case
videos_df = videos_df.drop_duplicates(subset=["video_id"])

out_path = "data/raw/week6/ted_tedx_videos.csv"
videos_df.to_csv(out_path, index=False)

print("Saved:", len(videos_df), "videos to", out_path)
videos_df.head()


Saved: 25491 videos to data/raw/week6/ted_tedx_videos.csv


,video_id,title,channel,published_at
0,iOTCsXd38Ng,What if plastic didn’t last forever? #TEDTalks,TED,2026-02-08T19:00:41Z
1,d1yfb93beSI,How to Introduce Yourself — and Get Hired | Re...,TED,2026-02-08T16:00:53Z
2,yrVtEkCC8uY,These ads could encourage people to think twic...,TED,2026-02-07T19:01:00Z
3,an6ZM0iCGQw,Let’s Build AI Data Centers in Space | Philip ...,TED,2026-02-06T16:00:07Z
4,bA9xGDsQADo,Today’s athletes ARE built different #TEDTalks,TED,2026-02-05T20:00:34Z


In [15]:
import os, glob
print("Folder exists?", os.path.exists("data/raw/week6"))
print("Files:", glob.glob("data/raw/week6/*"))


Folder exists? True
Files: ['data/raw/week6/ted_tedx_videos.csv']


## Week 6: Channel-Based Sampling Frame (TED & TEDx)

This section constructs the full population of TED and TEDx videos
and assigns each video to a publication-period time bin.

All subsequent comment scraping and sentiment analysis will operate
on this fixed channel-level sampling frame to ensure reproducibility
and comparability across time.


In [17]:
# Fix: ensure published_at is a proper datetime (not a string)
videos_df["published_at"] = pd.to_datetime(videos_df["published_at"], errors="coerce")

print("Null dates after conversion:", videos_df["published_at"].isna().sum())
print("Example:", videos_df["published_at"].dropna().iloc[0])


Null dates after conversion: 0
Example: 2026-02-08 19:00:41+00:00


In [18]:
# Define publication-period bins (same logic as earlier weeks)
def assign_time_bin(dt):
    year = dt.year
    if year < 2010:
        return "Pre-2010"
    elif year < 2020:
        return "2010–2019"
    else:
        return "2020+"

videos_df["time_bin"] = videos_df["published_at"].apply(assign_time_bin)

# Quick sanity check
videos_df.groupby(["channel", "time_bin"])["video_id"].nunique()


channel  time_bin 
TED      2010–2019     2620
         2020+         2300
         Pre-2010       571
TEDx     2020+        20000
Name: video_id, dtype: int64

In [19]:
# 3️⃣ Save the finalized sampling frame (Week 6 core artifact)

import os

out_path = "data/processed/week6/ted_tedx_video_sampling_frame.csv"
os.makedirs("data/processed/week6", exist_ok=True)

videos_df.to_csv(out_path, index=False)

print("Saved sampling frame to:", out_path)
print("Rows:", videos_df.shape[0])
print("Columns:", videos_df.columns.tolist())


Saved sampling frame to: data/processed/week6/ted_tedx_video_sampling_frame.csv
Rows: 25491
Columns: ['video_id', 'title', 'channel', 'published_at', 'time_bin']
